In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('loan_data.csv')
df.head()

,applicant_id,name,age,income,credit_score,loan_amount,employment_status,married,gender,approved
0,APP1170,Raj Smith,NaN,"$66,400",NaN,89100.0,Unemployed,No,Female,Rejected
1,APP1177,Omar Silva,25.0,"$75,700",645.0,228600.0,Employed,NaN,Male,Approved
2,APP1132,Mei Khan,24.0,"$26,400",580.0,5000.0,employed,Yes,Female,Approved
3,APP1193,Liam Ali,28.0,"$78,500",NaN,272600.0,Employed,No,Female,Approved
4,APP1102,Raj Kumar,35.0,NaN,573.0,159000.0,employed,N,Female,Rejected


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 212 entries, 0 to 211
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   applicant_id       212 non-null    object 
 1   name               212 non-null    object 
 2   age                194 non-null    float64
 3   income             202 non-null    object 
 4   credit_score       195 non-null    float64
 5   loan_amount        212 non-null    float64
 6   employment_status  201 non-null    object 
 7   married            200 non-null    object 
 8   gender             202 non-null    object 
 9   approved           212 non-null    object 
dtypes: float64(3), object(7)
memory usage: 16.7+ KB


In [6]:
df.describe()

,age,credit_score,loan_amount
count,194.000000,195.000000,212.000000
mean,43.448454,648.574359,152261.792453
std,20.945523,132.753971,85665.271670
min,-5.000000,0.000000,5000.000000
25%,31.000000,582.500000,88950.000000
50%,41.000000,652.000000,155800.000000
75%,54.000000,737.500000,207575.000000
max,200.000000,850.000000,448800.000000


In [7]:
df.isna().sum()

applicant_id          0
name                  0
age                  18
income               10
credit_score         17
loan_amount           0
employment_status    11
married              12
gender               10
approved              0
dtype: int64

In [9]:
df=df.drop(columns=['applicant_id','name'])
df.columns.tolist()

['age',
 'income',
 'credit_score',
 'loan_amount',
 'employment_status',
 'married',
 'gender',
 'approved']

In [10]:
df=df.drop_duplicates()

In [16]:
df['income']=(df['income'].astype('string').str.replace(r'[$,]', '',regex=True).str.strip())
df['income'] = pd.to_numeric(
    df['income'],
    errors='coerce'
)
df['income'].head().tolist()

[66400.0, 75700.0, 26400.0, 78500.0, nan]

In [17]:
df['gender'].value_counts(dropna=False)

Male      37
M         34
female    34
male      31
F         28
Female    26
NaN       10
Name: gender, dtype: int64

In [18]:
df['gender'] = df['gender'].astype(str).str.strip().str.lower().map(
    {'male': 'Male', 'm': 'Male', 'female': 'Female', 'f': 'Female'}
)
df['married'] = df['married'].astype(str).str.strip().str.lower().map(
    {'yes': 'Yes', 'y': 'Yes', 'no': 'No', 'n': 'No'}
)
emp_map = {
    'employed': 'Employed',
    'self employed': 'Self-Employed', 'self-employed': 'Self-Employed',
    'unemployed': 'Unemployed',
}
df['employment_status'] = df['employment_status'].astype(str).str.strip().str.lower().map(emp_map)


df['gender'].value_counts(dropna=False)

Male      102
Female     88
NaN        10
Name: gender, dtype: int64

In [19]:
import numpy as  np
df.loc[(df['age'] < 18) | (df['age'] > 100),'age']=np.nan
df.loc[(df['credit_score'] < 300) | (df['credit_score'] > 850),'credit_score']=np.nan

In [22]:
df.describe()

,age,income,credit_score,loan_amount
count,180.000000,188.000000,180.000000,200.000000
mean,42.327778,52789.361702,660.288889,151629.500000
std,12.861458,21143.955119,94.675836,86473.742375
min,21.000000,12000.000000,425.000000,5000.000000
25%,31.000000,37925.000000,584.000000,88250.000000
50%,41.000000,51150.000000,665.500000,157450.000000
75%,54.000000,68325.000000,731.750000,205700.000000
max,64.000000,104000.000000,850.000000,448800.000000


In [23]:
num_cols = ['age', 'income', 'credit_score', 'loan_amount']
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

for c in ['gender', 'employment_status', 'married']:
    df[c] = df[c].fillna(df[c].mode()[0])

In [24]:
df['approved'] = df['approved'].map({'Approved': 1, 'Rejected': 0})
df = pd.get_dummies(df, columns=['gender', 'employment_status', 'married'], drop_first=True)

df.head()

,age,income,credit_score,loan_amount,approved,gender_Male,employment_status_Self-Employed,employment_status_Unemployed,married_Yes
0,41.0,66400.0,665.5,89100.0,0,0,0,1,0
1,25.0,75700.0,645.0,228600.0,1,1,0,0,1
2,24.0,26400.0,580.0,5000.0,1,0,0,0,1
3,28.0,78500.0,665.5,272600.0,1,0,0,0,0
4,35.0,51150.0,573.0,159000.0,0,0,0,0,0


In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
df[num_cols].head()

,age,income,credit_score,loan_amount
0,-0.098160,0.670402,0.052355,-0.724918
1,-1.412433,1.125193,-0.176488,0.892336
2,-1.494575,-1.285690,-0.902087,-1.699909
3,-1.166007,1.262119,0.052355,1.402438
4,-0.591012,-0.075358,-0.980228,0.085448


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = df.drop(columns=['approved'])
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

print(f"Test accuracy: {model.score(X_test, y_test):.2f}")

Test accuracy: 0.75


In [27]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.78      0.78      0.78        23
           1       0.71      0.71      0.71        17

    accuracy                           0.75        40
   macro avg       0.74      0.74      0.74        40
weighted avg       0.75      0.75      0.75        40

